# CS383: Data Science and Machine Learning
## Lecture 7 — Linear Regression

*Dr. Thitima Srivatanakul*

### Guiding question
**How does a model turn a scatter of points into a single "best" line — and how do you tell the
difference between a model that found something real and one that's honestly telling you it found
nothing?**

### Learning objectives
By the end of this lecture, you should be able to:

- derive the least-squares solution for simple linear regression, and confirm it matches what
  scikit-learn's `LinearRegression` fits;
- fit and correctly interpret a multiple linear regression model's coefficients;
- compute and explain MAE, MSE, RMSE, and R² — including what each one does and doesn't tell you;
- use ablation (fitting a feature alone vs. combined) to figure out which feature is actually driving a
  model's R², and judge whether that's genuine signal or something to be suspicious of;
- read a residual plot, and recognize that "well-behaved residuals" and "a good model" are two
  different questions.

---

### Where this fits
Lecture 6 already prepped two datasets for exactly this moment: NYC 311's `resolution_time_hours`
(explicitly set aside as "a real regression target in Week 7") and restaurant inspection features
(encoded, scaled). Today we finally build the model — starting from the actual math, not just an API
call, so `LinearRegression()` stops being a black box. Fair warning going in: real civic data of the kind
this whole course uses is often *genuinely hard to predict* from the columns you happen to have, and
when a feature suddenly makes a model look great, that's exactly when you need to slow down and ask why.
This lecture builds both habits — noticing weak signal, and interrogating strong signal — using two real
datasets where each one shows up in turn.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. Part 1 builds intuition for what a regression line actually is — eyeballing a
trend, picking the best of several candidate lines, and seeing a real dataset where the honest answer is
"barely any trend at all." Part 2 works out the math behind "best fit," visually: what a residual is, why
squaring it matters, and a hands-on hunt for the least-squares line before the formula is revealed.

**[Open the "What Is Regression, Really?" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect07/regression_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Simple Linear Regression, From Scratch

Before letting scikit-learn do it for us, let's see exactly what "fitting a line" means.

### Setup — NYC 311 complaints

Same source Lecture 6 flagged as "a real regression target in Week 7": `resolution_time_hours` (how long
a complaint took to close), alongside `complaint_type`, `borough`, and the hour it was filed. We also
drop the handful of complaints logged under a non-standard `"Unspecified"` borough (too few to fit a
reliable coefficient for), and bucket every complaint type outside the 15 most common ones into
`"Other"` — with 100+ distinct complaint types in the raw data, some appearing only once, a full one-hot
encoding would otherwise let a single row masquerade as a pattern.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Deliberately built with
    # resolution time unrelated to complaint_type -- see Part 3 for why the real snapshot behaves
    # differently.
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

# Keep only complaints that actually closed (resolution_time_hours is defined), and only the 5
# standard NYC boroughs -- a handful of rows are tagged "Unspecified," too rare to fit a reliable
# coefficient for.
complaints_df = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)
standard_boroughs = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
complaints_df = complaints_df[complaints_df["borough"].isin(standard_boroughs)].reset_index(drop=True)

# complaint_type has 100+ distinct values in the real data, many with only a handful of rows -- group
# everything outside the 15 most common types into "Other" so a one-hot encoding doesn't end up
# treating a single rare row as if it were a reliable pattern.
top_types = complaints_df["complaint_type"].value_counts().nlargest(15).index
complaints_df["complaint_grouped"] = complaints_df["complaint_type"].where(
    complaints_df["complaint_type"].isin(top_types), "Other"
)

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "complaint_grouped", "borough", "hour_filed", "resolution_time_hours"]].head()

In [ ]:
plt.scatter(complaints_df["hour_filed"], complaints_df["resolution_time_hours"], alpha=0.15, s=10)
plt.ylim(0, 300)  # zoomed in purely for visibility -- more on the points above this line in Part 4
plt.xlabel("Hour complaint was filed (0-23)")
plt.ylabel("Resolution time (hours)")
plt.title("Resolution Time vs. Hour Filed")
plt.show()

Across the full 0-23 hour range, the cloud looks about the same height everywhere — no obvious upward or
downward drift as the hour changes. (The y-axis above is zoomed to the first 300 hours purely for
visibility; a small number of complaints take far longer than that to resolve, which Part 4 comes back to
directly.)

### The model: one line through the cloud

A simple linear regression model says the relationship between one input $x$ and output $y$ can be
approximated by a straight line:

$$\hat{y} = b_0 + b_1 x$$

- $\hat{y}$ ("y-hat") is the model's *predicted* resolution time — not the actual one.
- $b_0$ is the **intercept**: the predicted resolution time when `hour_filed` is exactly 0 (midnight).
- $b_1$ is the **slope**: how much the predicted resolution time changes for each additional hour later
  in the day a complaint is filed.

Every choice of $b_0$ and $b_1$ draws a different line. "Fitting" the model means picking the *one* line
that fits this data best — regardless of whether that best line turns out to be a good line.

### Defining "best": residuals and the cost function

For any candidate line, the **residual** for a single row is how far off that line's prediction was:

$$e_i = y_i - \hat{y}_i$$

A good line has small residuals across the board. To turn "small residuals across the board" into a
single number we can minimize, linear regression uses the **sum of squared residuals**:

$$SSR = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Why *squared*, instead of just summing the residuals directly? Two reasons: squaring makes every term
positive, so overestimates and underestimates can't cancel out — and it also gives a smooth function we
can minimize with calculus (next). It has a side effect worth remembering, too: squaring makes a residual
of 10 count 100x as much as a residual of 1 — large misses are punished disproportionately.

### Deriving the least-squares solution

We want the $b_0$ and $b_1$ that make $SSR$ as small as possible. Calculus tells us: at a minimum, the
*partial derivatives* of $SSR$ with respect to $b_0$ and $b_1$ are both zero. Working through that (you
don't need to reproduce this derivation yourself, but you should be able to follow it) gives two clean,
closed-form formulas:

$$b_1 = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b_0 = \bar{y} - b_1 \bar{x}$$

In words: the slope is the covariance between $x$ and $y$, divided by the variance of $x$. The intercept
just makes sure the line passes through the point $(\bar{x}, \bar{y})$ — the "center of mass" of the
data. Notice this formula has no opinion about whether $x$ and $y$ are actually related — if the
covariance in the numerator is near zero, it will honestly hand back a slope near zero.

In [ ]:
x = complaints_df["hour_filed"].values
y = complaints_df["resolution_time_hours"].values

x_bar = x.__________()
y_bar = y.__________()

b1_byhand = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar) ** 2)
b0_byhand = y_bar - b1_byhand * x_bar

print(f"By-hand intercept (b0): {b0_byhand:.4f}")
print(f"By-hand slope     (b1): {b1_byhand:.4f}")

In [ ]:
simple_model = __________()
simple_model.fit(complaints_df[["hour_filed"]], complaints_df["resolution_time_hours"])

print(f"scikit-learn intercept: {simple_model.intercept_:.4f}")
print(f"scikit-learn slope:     {simple_model.coef_[0]:.4f}")

Same numbers (up to rounding). `LinearRegression()` isn't doing anything mysterious — it's solving the
exact same minimization problem we just derived by hand, and it's just as honest about a near-zero
slope as our by-hand version was. Every regression model in this course builds on this same idea: define
a cost function, find the parameters that minimize it — whatever those parameters turn out to be.

In [ ]:
x_range = np.__________(0, 23, 100)
y_line = b0_byhand + b1_byhand * x_range

plt.scatter(complaints_df["hour_filed"], y, alpha=0.15, s=10, label="Actual complaints")
plt.plot(x_range, y_line, color="#D98C3F", linewidth=2, label="Fitted line")
plt.ylim(0, 300)
plt.xlabel("Hour complaint was filed (0-23)")
plt.ylabel("Resolution time (hours)")
plt.title("The Least-Squares Line")
plt.legend()
plt.show()

The fitted line is nearly flat — a slope of about half an hour of extra resolution time per hour later in
the day, essentially nothing next to typical resolution times. That's not a plotting mistake — it's the
honest least-squares answer for this particular $x$ and $y$. A flat line is still a real fitted model;
it's just a model whose best prediction barely changes no matter what you feed it.

---

## Part 2 — Multiple Linear Regression

Real problems rarely stop at one predictor. `borough` is sitting right there too, so let's add it
alongside `hour_filed`. `borough` is categorical, though, so before reaching for scikit-learn's
`ColumnTransformer` (Part 3 will), let's build the design matrix ourselves once, using
`pd.get_dummies(..., drop_first=True)` — dropping one category as the reference avoids a redundant
column that would otherwise make the matrix inversion below fail.

### From one line to a matrix equation

With two predictors, the model becomes:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2$$

and in general, with $p$ predictors:

$$\hat{y} = b_0 + b_1 x_1 + b_2 x_2 + \dots + b_p x_p$$

This is much easier to write — and to solve — in matrix form. Stack a column of 1s onto your feature
matrix (for the intercept), and the whole model becomes $\hat{y} = X\beta$, where $\beta$ is the vector of
all the $b$'s. Minimizing the same sum-of-squared-residuals cost function now has a closed-form matrix
solution, the **normal equation**:

$$\beta = (X^T X)^{-1} X^T y$$

This is the direct generalization of the two formulas from Part 1 — same idea, just written for many
predictors at once.

In [ ]:
borough_dummies = pd.get_dummies(complaints_df["borough"], drop_first=True).astype(float)
print("Reference category (dropped): BRONX")
print("Dummy columns:", borough_dummies.columns.tolist())

X_raw = pd.concat([complaints_df[["hour_filed"]], borough_dummies], axis=1).values.astype(float)
X_design = np.column_stack([np.ones(len(X_raw)), X_raw])  # add intercept column of 1s
y_vals = complaints_df["resolution_time_hours"].values

beta_byhand = np.linalg.__________(X_design.T @ X_design) @ X_design.T @ y_vals

print("By-hand coefficients [intercept, hour_filed, BROOKLYN, MANHATTAN, QUEENS, STATEN ISLAND]:")
print(beta_byhand.round(4))

In [ ]:
X_features = pd.concat(
    [complaints_df[["hour_filed"]], pd.get_dummies(complaints_df["borough"], drop_first=True).astype(float)],
    axis=1,
)

multi_model = LinearRegression()
multi_model.__________(X_features, complaints_df["resolution_time_hours"])

print(f"scikit-learn intercept: {multi_model.intercept_:.4f}")
print(f"scikit-learn coefficients: {multi_model.coef_.round(4)}")

Same numbers again. (In practice, scikit-learn doesn't literally invert a matrix like we just did —
matrix inversion is numerically unstable for larger problems, so it uses a more stable least-squares
solver internally. The *math it's solving* is identical either way.)

### Interpreting the coefficients

Each coefficient is a **partial effect**: holding everything else fixed, how much does the predicted
resolution time change for a one-unit increase in that feature? `hour_filed`'s coefficient (≈0.42) says
each hour later in the day a complaint is filed adds under half an hour to the predicted resolution time
— tiny. The four borough coefficients are all relative to `BRONX`, the category we dropped as the
reference: Brooklyn adds about 9 hours, Staten Island about 11, Queens about 13, and Manhattan a full 37
hours, all holding `hour_filed` fixed.

Those numbers are believable — a borough-level gap in how long city agencies take to close out
complaints is a plausible real-world pattern, not an obvious red flag. But notice what "believable
coefficients" doesn't tell you: whether this model, overall, is any good at predicting resolution time.
That's a completely different question — it's what R² measures — and Part 3 checks it directly.

---

## Part 3 — Evaluating a Regression Model

An $R^2$, an MAE, an RMSE — the Iris sneak peek used these informally. Here's what each one actually
measures, and when to reach for which — evaluated properly on a held-out test set this time, rather than
fit-and-eyeball the way Parts 1-2 did. We'll also bring back the one feature Part 1-2 left out —
`complaint_grouped` — and see what it does to the picture.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    complaints_df[["hour_filed", "borough", "complaint_grouped"]],
    complaints_df["resolution_time_hours"],
    test_size=__________, random_state=383,
)

preprocessor_311 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough", "complaint_grouped"]),
])
X_train_ready = preprocessor_311.fit_transform(X_train)
X_test_ready = preprocessor_311.transform(X_test)

eval_model = LinearRegression()
eval_model.fit(X_train_ready, y_train)
y_pred = eval_model.predict(X_test_ready)

print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))

### Mean Absolute Error (MAE)

$$MAE = \frac{1}{n}\sum |y_i - \hat{y}_i|$$

The average size of a miss, in the original units (hours, here) — the most directly interpretable
metric. It treats every error in proportion to its size, so one huge miss doesn't dominate the number.

In [ ]:
mae = __________(y_test, y_pred)
print(f"MAE: {mae:.2f} hours")

### Mean Squared Error (MSE) and Root Mean Squared Error (RMSE)

$$MSE = \frac{1}{n}\sum (y_i - \hat{y}_i)^2 \qquad RMSE = \sqrt{MSE}$$

Squaring the errors (same reasoning as Part 1's cost function) makes large misses count
disproportionately more. RMSE takes the square root at the end, bringing the units back to the original
scale (hours, not hours²) so it's directly comparable to MAE — and $RMSE \geq MAE$ always. The gap
between them tells you something: a big gap means a few large errors are dragging the average up; a small
gap means errors are fairly uniform in size.

In [ ]:
mse = __________(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f} hours")

### R² (coefficient of determination)

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

$SS_{res}$ is the same sum-of-squared-residuals from Part 1's cost function — how much error the model
actually made. $SS_{tot}$ is how much error you'd make with the simplest possible model: always
predicting the mean, $\bar{y}$. $R^2$ is the fraction of that baseline error your model *eliminated*.

- $R^2 = 1$: perfect predictions.
- $R^2 = 0$: your model does no better than just guessing the mean every time.
- $R^2 < 0$: your model does *worse* than guessing the mean — a real possibility on a bad fit or a badly
  mismatched test set, not just a theoretical edge case.

In [ ]:
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2_byhand = 1 - ss_res / ss_tot

r2 = __________(y_test, y_pred)

print(f"By-hand R²: {r2_byhand:.4f}")
print(f"sklearn R²: {r2:.4f}")

### Which feature is actually doing the work?

R² tells you how much of the variation in `resolution_time_hours` this three-feature model explains
overall. It doesn't tell you how much each feature is contributing on its own — and Part 2's coefficients
already hinted that `hour_filed` and `borough` alone might not be carrying much. Let's check, by fitting
progressively larger models on the same train/test split and comparing R² at each step.

In [ ]:
model_hour_only = LinearRegression().fit(X_train[["hour_filed"]], y_train)
r2_hour_only = r2_score(y_test, model_hour_only.predict(X_test[["hour_filed"]]))

preprocessor_hb = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["borough"]),
])
X_train_hb = preprocessor_hb.fit_transform(X_train[["hour_filed", "borough"]])
X_test_hb = preprocessor_hb.transform(X_test[["hour_filed", "borough"]])
model_hour_borough = LinearRegression().fit(X_train_hb, y_train)
r2_hour_borough = __________(y_test, model_hour_borough.predict(X_test_hb))

print(f"hour_filed alone:                     R² = {r2_hour_only:.4f}")
print(f"hour_filed + borough:                 R² = {r2_hour_borough:.4f}")
print(f"hour_filed + borough + complaint_type: R² = {r2:.4f}")

In [ ]:
complaints_df.groupby("complaint_grouped")["resolution_time_hours"].__________().sort_values()

`hour_filed` alone and `hour_filed` + `borough` both land at essentially R² = 0 — consistent with Part 2's
own coefficients, which were believable but never obviously doing much. But adding `complaint_grouped`
jumps R² to about 0.17. That's a real, meaningful chunk of the variation in resolution time, not a
rounding artifact.

Is this leakage, the way it might look at first glance? No — and it's worth being explicit about why not.
`complaint_type` is known the moment a complaint is filed, before any resolution happens; it isn't
computed from `resolution_time_hours` the way a feature in a different case study (coming up) turns out
to be computed from its target. The median-by-type table above makes the pattern obvious: quick-response
types like `Noise - Commercial` or `Illegal Fireworks` typically close in about an hour, while
infrastructure complaints like `UNSANITARY CONDITION`, `PLUMBING`, or `HEAT/HOT WATER` typically take
days to months. Different city agencies, with very different response-time norms, handle these — and
`complaint_type` is effectively standing in for "which agency, with what backlog." That's genuine, useful,
non-circular signal.

The habit that got you here matters more than this specific result: whenever adding one feature causes a
disproportionate jump in R², don't just accept the higher number — ask *why*, and check whether the
answer holds up. Here, it does. Coming up, it won't.

---

## Part 4 — Checking the Model: Residual Plots

A single R² number can hide a lot — but a residual plot can also mislead you in the *other* direction if
you're not careful about what it can and can't tell you.

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.3, s=12)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted resolution time (hours)")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted Resolution Time")
plt.show()

Expect a lopsided, fan-shaped spread here: most predictions cluster in a fairly narrow band, but a long
tail of complaints took far longer to resolve than the model expected — some residuals run past +4,700
hours (about 6 months), while the model is never off by more than about 560 hours on the low side. That
asymmetry is a direct fingerprint of `resolution_time_hours` itself, which we'll look at directly next:
most complaints close quickly, a smaller number take a very long time, and a straight line fit to the raw
hours can't bend to match that shape. This is a different, additional problem on top of — not instead of
— the genuine signal the ablation above just found: R² of 0.17 means the model is doing real work, and
still leaving a lot on the table.

In [ ]:
plt.__________(residuals, bins=30, color="#2E5C8A", edgecolor="white")
plt.xlabel("Residual")
plt.title("Distribution of Residuals")
plt.show()

Residuals should be roughly centered on 0 to indicate no systematic bias, and this one is — but
"centered" and "symmetric" are different properties, and this histogram is far from symmetric: a tall
spike of small residuals near zero, and a long thin tail stretching out toward the extreme positive side
(some complaints resolve hundreds of hours slower than predicted; almost none resolve anywhere near that
much faster). That lopsidedness mirrors the target's own distribution almost exactly — worth checking
directly, next.

In [ ]:
plt.__________(y_train, bins=40, color="#2E5C8A", edgecolor="white")
plt.xlabel("Resolution time (hours)")
plt.title("Distribution of Resolution Time (Training Set)")
plt.show()

print(f"Mean:   {y_train.mean():.1f} hours")
print(f"Median: {y_train.median():.1f} hours")

A mean of about 78 hours next to a median of about 2 hours is about as clear a right-skew signature as
you'll see: half of all complaints in the training set close within about 2 hours, but a long tail of slow
ones (some taking weeks or months) drags the average up to nearly 35x the typical case. Predicting a
target shaped like this directly, with a method that minimizes *squared* error, means those rare extreme
cases exert an outsized pull on the fitted line — exactly the fan-shaped residual pattern you just saw.

### A common fix: log-transform the target

Applying `log1p()` (log of $1+x$, so it stays defined at 0) compresses that long right tail, making the
target's distribution look a lot closer to symmetric — often a better match for what a straight line can
actually capture. "Often" is doing real work in that sentence, though — let's actually check, rather than
assume it helps here.

In [ ]:
y_train_log = np.__________(y_train)
y_test_log = np.__________(y_test)

model_log = LinearRegression()
model_log.fit(X_train_ready, y_train_log)
y_pred_log = model_log.predict(X_test_ready)

r2_log = r2_score(y_test_log, y_pred_log)
print(f"R² on raw hours:              {r2:.4f}")
print(f"R² on log-transformed target: {r2_log:.4f}")

That's a big jump — R² roughly triples, from about 0.17 on the raw scale to about 0.57 on the
log-transformed scale. Some caution is still warranted: these two R² values are measured on different
scales (hours vs. log-hours), so they're not strictly the same yardstick, and a higher R² on log-hours
doesn't automatically mean better predictions once you convert back to real hours. But a jump this large,
on a target this skewed, is a real signal that the transform is helping the model fit the data it's
actually built to fit — squared error, which heavily-skewed targets abuse. If you need predictions back in
real hours, apply `np.expm1()` to reverse the transform, and re-check MAE/RMSE on that original scale
before claiming victory — a model that fits log-hours well doesn't automatically translate into small
errors in hours.

---

## Part 5 — A Second, Trickier Case: Restaurant Inspection Scores

311 just showed you a big jump in R² that turned out to be genuine. Now here's a second real dataset —
NYC restaurant inspections — where a similarly big jump shows up for a very different reason. Same tools,
same habit of asking "why": let's see if it holds up this time.

### Setup — NYC restaurant inspections

Same source as Lecture 6, rebuilt at the row level (one row per violation citation) rather than
collapsed into groups — we want `is_critical` and `grade` to describe an actual real record, not an
artifact of how rows happened to get grouped.

In [ ]:
try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade"]).reset_index(drop=True)

    # One row here is one violation citation, not one full inspection -- be careful not to call this
    # "violations per inspection," since a single inspection can contribute more than one row.
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})

    # grade also includes non-letter administrative statuses (N = not yet graded, Z = grade
    # pending, P = grade pending issued on reopening) that have no ordinal meaning -- drop those
    # so grade_ord never contains NaN wherever it's used as a model feature below.
    inspections_df = inspections_df.dropna(subset=["grade_ord"]).reset_index(drop=True)
    live_restaurants = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room. Deliberately built with
    # score unrelated to is_critical/grade -- that's what the real shared snapshot actually shows too.
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    is_critical = rng.integers(0, 2, size=n)
    grade = rng.choice(["A", "B", "C"], size=n, p=[0.6, 0.25, 0.15])
    score = rng.integers(0, 71, size=n)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    inspections_df["grade_ord"] = inspections_df["grade"].map({"A": 0, "B": 1, "C": 2})
    live_restaurants = False

print(f"{'Shared snapshot' if live_restaurants else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_description", "score", "grade", "is_critical"]].head()

### Same technique, new features

Same normal equation as Part 2 — $\beta = (X^TX)^{-1}X^Ty$ — just with two different columns this time:
`is_critical` and `grade_ord` (both already numeric, from Lecture 6's ordinal encoding, so no one-hot
detour needed).

In [ ]:
X_raw = inspections_df[["is_critical", "grade_ord"]].values
X_design = np.column_stack([np.ones(len(X_raw)), X_raw])  # add intercept column of 1s
y_vals = inspections_df["score"].values

beta_byhand = np.linalg.__________(X_design.T @ X_design) @ X_design.T @ y_vals

print("By-hand coefficients [intercept, is_critical, grade_ord]:")
print(beta_byhand.round(4))

In [ ]:
multi_model = LinearRegression()
multi_model.__________(inspections_df[["is_critical", "grade_ord"]], inspections_df["score"])

print(f"scikit-learn intercept: {multi_model.intercept_:.4f}")
print(f"scikit-learn coefficients: {multi_model.coef_.round(4)}")

Same numbers again, for the same reason as Part 2: scikit-learn solves the identical minimization
problem, just via a more numerically stable solver than a literal matrix inversion.

### Interpreting the coefficients

Just like Part 2, each coefficient is a **partial effect**: holding every other feature fixed, how much
does the predicted score change for a one-unit increase in that feature? Look at the actual sizes above
before moving on — they are not close to each other. `is_critical`'s coefficient is under 1 point.
`grade_ord`'s is around 15 points *per grade step* — over an order of magnitude larger, on a feature that
only ever takes the values 0, 1, or 2.

That gap is worth stopping on, not skimming past. One of these two features is apparently doing almost
all of the work in this model, and the other barely registers. Whenever a predictor's effect looks that
outsized, the right reflex is to ask *why*, before trusting the number — is this feature really carrying
independent information about the target, or is something else going on? The evaluation below digs into
exactly that question.

This is also a good moment for a caution that applies regardless of coefficient size: a coefficient
describes the model's fitted line, not a causal claim. A large coefficient doesn't by itself prove that
`grade_ord` *causes* the score it's paired with — it only describes the pattern this particular line
captured in this particular data. Sometimes, as you're about to see, that pattern has a much more mundane
explanation than "this feature genuinely predicts the outcome."

### Evaluating this model

You already know what MAE, RMSE, and R² measure from Part 3 — let's just compute them here, split
honestly into train/test first.

In [ ]:
X_scores = inspections_df[["is_critical", "grade_ord"]]
y_scores = inspections_df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X_scores, y_scores, test_size=__________, random_state=383
)

eval_model = LinearRegression()
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = __________(y_test, y_pred)

print(f"MAE:  {mae:.2f} points")
print(f"RMSE: {rmse:.2f} points")
print(f"R²:   {r2:.4f}")

### Which feature is actually doing the work?

R² tells you how much of the variation in `score` the two-feature model explains overall. It doesn't tell
you how much each feature is contributing on its own. Given how lopsided the coefficients above looked,
that's worth checking directly — fit three separate models on the same train/test split (`is_critical`
alone, `grade_ord` alone, and both together) and compare their R² side by side.

In [ ]:
model_critical_only = LinearRegression().fit(X_train[["is_critical"]], y_train)
r2_critical_only = r2_score(y_test, model_critical_only.predict(X_test[["is_critical"]]))

model_grade_only = LinearRegression().fit(X_train[["grade_ord"]], y_train)
r2_grade_only = __________(y_test, model_grade_only.predict(X_test[["grade_ord"]]))

print(f"is_critical alone:       R² = {r2_critical_only:.4f}")
print(f"grade_ord alone:         R² = {r2_grade_only:.4f}")
print(f"is_critical + grade_ord: R² = {r2:.4f}")

In [ ]:
inspections_df.groupby("grade")["score"].__________()

Sit with that gap for a second instead of rushing past it. `is_critical` alone gets an R² of about 0.02 —
genuinely near zero, a real and useful finding: whether a violation was critical tells you almost nothing
about the inspection score by itself. But `grade_ord` alone gets an R² of about 0.76, and adding
`is_critical` to it barely moves that number. Almost all of the combined model's explanatory power was
coming from `grade_ord` alone.

That's not `grade_ord` being a great *predictor* of score in the usual sense — it's `grade_ord` being
almost the same information as `score`, restated. NYC assigns letter grades by directly thresholding the
inspection score: roughly 0–13 points is an A, 14–27 is a B, and 28+ is a C — and the mean score by grade
above shows exactly that separation (A ≈ 10, B ≈ 22, C ≈ 42). So `grade_ord` isn't an independent signal
about the restaurant that happens to correlate with score — it's a coarser, bucketed *version* of the
score, computed by the same city agency from the same inspection. Feeding it into a model that predicts
score is close to feeding the model a rounded-off copy of the answer.

This is **target leakage**: a feature that is itself derived from (or nearly equivalent to) the target,
so a model built with it looks far more accurate than it actually is at the thing you presumably care
about — predicting an outcome from genuinely independent information. It's one of the most common ways a
model quietly cheats without anyone changing a line of the fitting code. The fix isn't a fancier
algorithm; it's asking, for every candidate feature, "could I have known this *before* the outcome I'm
trying to predict, and is it computed independently of that outcome?" `is_critical` passes that test here.
`grade_ord` does not.

Compare this to 311's `complaint_grouped`: both features caused a big jump in R² when added. One held up
under scrutiny — a real, causally plausible mechanism, known before the outcome. The other didn't — a
feature computed from the very thing it's "predicting." Same jump, opposite explanations, which is exactly
why the jump alone was never enough to trust on its own.

### Checking the model: residual plots

A single R² number can hide a lot — and, as you're about to see, a residual plot can hide something too.

In [ ]:
residuals = y_test - __________

plt.scatter(y_pred, residuals, alpha=0.4, s=15)
plt.axhline(0, color="#9AA5B1", linestyle="--")
plt.xlabel("Predicted score")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residuals vs. Predicted Score")
plt.show()

Because `is_critical` and `grade_ord` only ever take 2 and 3 distinct values respectively, this model can
only ever produce a handful of distinct predictions — you should see the dots cluster into exactly six
thin vertical bands, one per combination of the two features. Within each band, though, notice the spread
is still substantial (residuals ranging roughly ±40 points in places) — that's the leftover variation in
`score` that `grade_ord`'s coarse buckets can't capture, even though `grade_ord` is derived from `score`
in the first place.

**This plot cannot tell you about leakage.** A residual plot only checks whether the errors left over show
an obvious shape the model missed — it has no way to reveal that one of your features was built from your
target. This model can look clean here (flat, centered on zero, no funnel or curve) *and* still be
leaking, because a residual plot and a feature-provenance check answer two completely different
questions. Leakage lives upstream, in how the features were constructed — you have to go looking for it
directly, the way the ablation above just did, not wait for a diagnostic plot to surface it on its own.

In [ ]:
plt.__________(residuals, bins=30, color="#2E5C8A", edgecolor="white")
plt.xlabel("Residual")
plt.title("Distribution of Residuals")
plt.show()

Roughly symmetric and centered around 0 is what you're checking for here, and it holds up fine. But hold
the two facts side by side: R² of about 0.77 says this model explains most of the variation in `score` —
and yet the residuals here still spread out by tens of points in either direction. That's consistent, not
contradictory: R² of 0.77 means a *lot* of variation is explained, not *all* of it, and each of the six
predicted values in the scatter plot above is really standing in for a wide range of real inspections that
happen to share the same `is_critical`/`grade_ord` combination. A high R² driven by a leaked feature can
still leave real, sizable errors on individual predictions — accuracy on average and accuracy on any one
case are not the same promise.

### Two datasets, two different reasons to double-check

311 gave you a case where a big jump in R² was genuine — `complaint_grouped` really does carry
information about resolution time, because different types of complaints go to different agencies with
different response times. The restaurant data gave you a case where a big jump in R² was an illusion —
`grade_ord` looks predictive only because it's built from the target. Both times, the tool that told them
apart was the same: don't trust a coefficient or an R² at face value, fit the pieces separately, and ask
why. That habit — not any specific formula — is the actual takeaway of this lecture.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect07_regression_exercise.ipynb`.

---

## Part 6 — Cheat Sheet

| Task | Code |
|---|---|
| Fit a linear regression | `LinearRegression().fit(X_train, y_train)` |
| Predict | `model.predict(X_test)` |
| Coefficients / intercept | `model.coef_`, `model.intercept_` |
| MAE | `mean_absolute_error(y_test, y_pred)` |
| MSE / RMSE | `mean_squared_error(y_test, y_pred)`, then `np.sqrt(...)` |
| R² | `r2_score(y_test, y_pred)` |
| Residuals | `y_test - y_pred` |
| Ablation (which feature matters?) | fit on one feature at a time, compare R² |
| Compress a right-skewed target | `np.log1p(y)`, reverse with `np.expm1(...)` |

---

## Part 7 — Key Terms

- **Simple linear regression**: predicting a continuous target from one feature, using a straight line.
- **Multiple linear regression**: predicting a continuous target from several features at once.
- **Residual**: the difference between an actual value and the model's prediction, $y_i - \hat{y}_i$.
- **Least squares**: the method of choosing model parameters that minimize the sum of squared residuals.
- **Normal equation**: the closed-form matrix solution for the least-squares coefficients,
  $\beta = (X^TX)^{-1}X^Ty$.
- **MAE / MSE / RMSE**: average error size, in original units (MAE, RMSE) or squared units (MSE); RMSE
  and MSE penalize large errors more than MAE does.
- **R² (coefficient of determination)**: the fraction of the target's variance the model explains,
  relative to always predicting the mean. Can be near zero, or even negative, on real data.
- **Residual plot**: a plot of residuals against predicted values, used to check whether a model's
  errors are patternless (good) or systematically patterned (a sign the model is missing something) —
  a different question from whether the model explains much of the outcome.
- **Heteroscedasticity**: when a model's errors have different spread across the range of predictions —
  a fan-shaped residual plot is the visual sign of it.
- **Right-skewed target**: a target with a long tail of unusually large values; mean well above median
  is a quick sign of it, and it can make squared-error metrics and a plain linear fit misleading.
- **Target leakage**: when a feature is itself derived from (or nearly equivalent to) the target, so a
  model built with it looks far more accurate than it genuinely is. Ablation (comparing a feature's
  contribution alone vs. combined with others) is one way to catch it; a residual plot cannot.